In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor

from torch.utils.data import Dataset

In [2]:
import yaml
import numpy as np
import cv2
import os

## Model Classes

In [3]:
class ConvBlock(nn.Module):
    def __init__(self, inChannels, outChannels, kernelSize=3, stride=1, padding=1):
        """Conv -> BatchNorm -> ReLU"""
    
        super(ConvBlock, self).__init__()
        self.convolution = nn.Conv2d(inChannels, outChannels, kernelSize, stride, padding)
        self.batchNorm = nn.BatchNorm2d(outChannels)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.batchNorm(self.convolution(x)))

In [4]:
class ModelBackbone(nn.Module):
    def __init__(self):
        """Add together conv"""
        super(ModelBackbone, self).__init__()
        self.layers = nn.Sequential(
            ConvBlock(3, 32, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            ConvBlock(32, 64, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
            ConvBlock(64, 128, kernelSize=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
        )

    def forward(self, x):
        return self.layers(x)

In [5]:
class ModelHead(nn.Module):
    def __init__(self, gridSize, numClasses, numAnchors):
        """Predicts bbox, conf, cls"""
        super(ModelHead, self).__init__()
        self.gridSize = gridSize
        self.numClasses = numClasses
        self.numAnchors = numAnchors

        self.detector = nn.Conv2d(128, self.numAnchors * (5 + self.numClasses), kernel_size=1)
        
    def forward(self, x):
        pred = self.detector(x)
        pred = pred.permute(0, 2, 3, 1).contiguous()

        batchSize, h, w, _ = pred.shape

        pred = pred.view(batchSize, h, w, self.numAnchors, 5 + self.numClasses)

        return pred

In [6]:
class TLDetectionModel(nn.Module):
    def __init__(self, gridSize=7, numClasses=20, numAnchors=3):
        super(TLDetectionModel, self).__init__()
        self.backbone = ModelBackbone()
        self.head = ModelHead(gridSize, numClasses, numAnchors)

    def forward(self, x):
        features = self.backbone(x)
        predictions = self.head(features)
        return predictions

model = TLDetectionModel(gridSize=7, numClasses=3, numAnchors=3)
print(model)

TLDetectionModel(
  (backbone): ModelBackbone(
    (layers): Sequential(
      (0): ConvBlock(
        (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (2): ConvBlock(
        (convolution): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ConvBlock(
        (convolution): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()


## Preprocessing + Load Data

In [7]:
# Loss function
def lossFunc(predictions, targets, numClasses, lambdaCoord=5, lambdaNoObj = 0.5):
    """
    Computes YOLO loss.
    - predictions: Predicted tensor.
    - targets: Ground truth tensor.
    """

    predBoxes = predictions[..., :4]
    predConf = predictions[..., 4]
    predClasses = predictions[..., 5:]
    targetBoxes = targets[..., :4]
    targetConf = targets[..., 4]
    targetClasses = targets[..., 5:]

    bce = torch.nn.BCEWithLogitsLoss()

    
    objMask = targetConf == 1
    noObjMask = targetConf == 0
    

    boxLoss = lambdaCoord * torch.sum((predBoxes[objMask] - targetBoxes[objMask]) ** 2)
    
    objLoss = bce(predConf[objMask], targetConf[objMask])
    noObjLoss = lambdaNoObj * bce(predConf[noObjMask], targetConf[noObjMask])
    
    classLoss = bce(predClasses[objMask], targetClasses[objMask])
    

    totalLoss = (boxLoss + objLoss + noObjLoss + classLoss) / predictions.size(0)
    return totalLoss
    
    

In [8]:
class DatasetHelper(Dataset):
    def __init__(self, imgDir, labelDir, transforms=None):
        self.imgDir = imgDir
        self.labelDir = labelDir
        self.transforms = transforms
        self.images = [
            f for f in os.listdir(self.imgDir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        imgPath = os.path.join(self.imgDir, self.images[idx])
        labelPath = os.path.join(self.labelDir, self.images[idx].replace(".png", ".txt"))

        image = cv2.imread(imgPath)

        if image is None:
            print(f"ERROR: Failed to load image → {imgPath}")
            raise FileNotFoundError(imgPath)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        with open(labelPath, "r") as f:
            for line in f.readlines():
                classLabel, x, y, w, h = map(float, line.strip().split())
                boxes.append([classLabel, x, y, w, h])

        if self.transforms:
            image = self.transforms(image)
            
        return image, torch.tensor(boxes)


In [9]:
def collateFn(batch):
    images = []
    targets = []

    for img, target in batch:
        images.append(img)
        targets.append(target)

    images = torch.stack(images, 0)  
    return images, targets           

In [10]:
trainDataset = DatasetHelper(imgDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/train/images", labelDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/train/labels", transforms=ToTensor())
trainLoader = DataLoader(trainDataset, batch_size=8, num_workers=0, shuffle=True, pin_memory=True, collate_fn=collateFn)

valDataset = DatasetHelper(imgDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/val/images", labelDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/val/labels", transforms=ToTensor())
valLoader = DataLoader(valDataset, batch_size=8, num_workers=0, shuffle=True, pin_memory=True, collate_fn=collateFn)

testDataset = DatasetHelper(imgDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/test/images", labelDir="C:/PROJECTS/TrafficLightDetectionTutorial/TrainData/test/labels", transforms=ToTensor())
testLoader = DataLoader(testDataset, batch_size=8, num_workers=0, shuffle=True, pin_memory=True, collate_fn=collateFn)

In [11]:

def buildTarget(target, gridSize, numAnchors, numClasses):
    gridH, gridW = gridSize

    targetTensor = torch.zeros(
        (gridH, gridW, numAnchors, 5 + numClasses),
        device=device
    )

    for box in target:
        cls, x, y, w, h = box

        gridX = int(x * gridW)
        gridY = int(y * gridH)

        # prevent edge overflow
        gridX = min(gridX, gridW - 1)
        gridY = min(gridY, gridH - 1)

        anchor = 0

        cellX = x * gridW - gridX
        cellY = y * gridH - gridY

        targetTensor[gridY, gridX, anchor, 0:4] = torch.tensor(
            [cellX, cellY, w, h],
            device=device
        )

        targetTensor[gridY, gridX, anchor, 4] = 1
        targetTensor[gridY, gridX, anchor, 5 + int(cls)] = 1

    return targetTensor

In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learningRate = 1e-4 #0.0001
numEpochs = 600
numClasses = 3

## Run Training

In [14]:
model = TLDetectionModel(gridSize=64, numClasses=numClasses, numAnchors=3)
optimizer = optim.Adam(model.parameters(), lr=learningRate)
criterion = lossFunc

model = model.to('cuda')

for module in model.modules():
    module.to('cuda')

print("Begin")

bestEpoch = [0, 100000000]

for epoch in range(numEpochs):
    model.train()

    epochLoss = 0
    subCount = 0

    for images, targets in trainLoader:
        images = images.to('cuda')
        targets = [t.to('cuda') for t in targets]

        predictions = model(images)

        _, gridH, gridW, _, _ = predictions.shape

        targetGrid = torch.stack([buildTarget(t, (gridH, gridW), 3, numClasses) for t in targets]).to(predictions.device)

        loss = criterion(predictions, targetGrid, numClasses=numClasses)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epochLoss += loss.item()
        subCount += 1

        if subCount % 10 == 0:
            print(f"Sub {subCount}. Loss: {loss.item():.4f}")

    averageTrainLoss = epochLoss / subCount

    model.eval()

    validationLoss = 0
    validationCount = 0

    with torch.no_grad():
        for images, targets in valLoader:
            images = images.to('cuda')
            targets = [t.to('cuda') for t in targets]

            predictions = model(images)

            _, gridH, gridW, _, _ = predictions.shape

            targetGrid = torch.stack([buildTarget(t, (gridH, gridW), 3, numClasses) for t in targets]).to(predictions.device)

            loss = criterion(predictions, targetGrid, numClasses=numClasses)

            validationLoss += loss.item()
            validationCount += 1

    averageValidationLoss = validationLoss / validationCount

    print(
        f">>> Epoch {epoch + 1}/{numEpochs} "
        f"| Train Loss: {averageTrainLoss:.4f} "
        f"| Validation Loss: {averageValidationLoss:.4f} <<<"
    )

    # Save best epoch based on validation loss
    if averageValidationLoss < bestEpoch[1]:
        print(f"New Best: {averageValidationLoss:.4f} < {bestEpoch[1]:.4f}")
        bestEpoch[1] = averageValidationLoss
        bestEpoch[0] = epoch

print(
    f"Best Epoch: {bestEpoch[0] + 1} "
    f"with validation loss of {bestEpoch[1]:.4f}"
)

Begin
Sub 10. Loss: 62.4663
Sub 20. Loss: 10.1220
Sub 30. Loss: 8.6394
Sub 40. Loss: 6.4204
>>> Epoch 1/600 | Train Loss: 24.4523 | Validation Loss: 3.3920 <<<
New Best: 3.3920 < 100000000.0000
Sub 10. Loss: 2.3884
Sub 20. Loss: 5.5848
Sub 30. Loss: 4.4562
Sub 40. Loss: 2.6404
>>> Epoch 2/600 | Train Loss: 4.2403 | Validation Loss: 3.6582 <<<
Sub 10. Loss: 3.0257
Sub 20. Loss: 2.9123
Sub 30. Loss: 3.2889
Sub 40. Loss: 3.6334
>>> Epoch 3/600 | Train Loss: 2.8889 | Validation Loss: 3.2302 <<<
New Best: 3.2302 < 3.3920
Sub 10. Loss: 1.7146
Sub 20. Loss: 1.6945
Sub 30. Loss: 1.5336
Sub 40. Loss: 1.1753
>>> Epoch 4/600 | Train Loss: 2.2312 | Validation Loss: 2.7408 <<<
New Best: 2.7408 < 3.2302
Sub 10. Loss: 1.4710
Sub 20. Loss: 2.1645
Sub 30. Loss: 2.2214
Sub 40. Loss: 1.4774
>>> Epoch 5/600 | Train Loss: 1.8013 | Validation Loss: 2.4679 <<<
New Best: 2.4679 < 2.7408
Sub 10. Loss: 1.4705
Sub 20. Loss: 1.5619
Sub 30. Loss: 2.5615
Sub 40. Loss: 1.3617
>>> Epoch 6/600 | Train Loss: 1.6321 | V

In [15]:
torch.save(model.state_dict(), "model.pth")

## Testing

In [13]:
model = TLDetectionModel(gridSize=64, numClasses=3, numAnchors=3)
model.load_state_dict(torch.load("model.pth"))
model.to('cuda')
model.eval()

TLDetectionModel(
  (backbone): ModelBackbone(
    (layers): Sequential(
      (0): ConvBlock(
        (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (2): ConvBlock(
        (convolution): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()
      )
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ConvBlock(
        (convolution): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (batchNorm): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU()


In [15]:
model.eval()
criterion = lossFunc

testLoss = 0
testCount = 0

with torch.no_grad():
    for images, targets in testLoader:
        images = images.to('cuda')
        targets = [t.to('cuda') for t in targets]

        predictions = model(images)

        _, gridH, gridW, _, _ = predictions.shape

        targetGrid = torch.stack([
            buildTarget(t, (gridH, gridW), 3, numClasses)
            for t in targets
        ]).to(predictions.device)

        loss = criterion(predictions, targetGrid, numClasses=numClasses)

        testLoss += loss.item()
        testCount += 1

averageTestLoss = testLoss / testCount

print(f"Test Loss: {averageTestLoss:.4f}")

Test Loss: 0.2865
